# mini_gpt — Xây GPT từ đầu / Build a GPT from scratch

**Notebook song ngữ Việt–Anh** dựa trên nanoGPT của Andrej Karpathy.

Bilingual VN–EN notebook based on Karpathy's nanoGPT.

Chạy lần lượt từng cell từ trên xuống (Shift+Enter). CPU cũng chạy được trong vài phút.

Run each cell top to bottom (Shift+Enter). Works on CPU in a few minutes.

---
**4 bước / 4 steps:** dữ liệu → mô hình → huấn luyện → sinh chữ · data → model → train → generate

## 0. Cài đặt / Setup

Cần PyTorch. Nếu chưa có, bỏ dấu `#` ở dòng dưới. / Needs PyTorch; uncomment below if missing.

In [ ]:
!pip install torch
import torch, torch.nn as nn
from torch.nn import functional as F
print('torch', torch.__version__)

## 1. Siêu tham số / Hyperparameters

Các con số điều khiển kích thước & tốc độ học. Muốn kết quả tốt hơn: tăng `n_layer`, `n_embd`, `max_iters` (chậm hơn).

Knobs controlling size & training. For better output: raise `n_layer`, `n_embd`, `max_iters` (slower).

In [ ]:
batch_size = 32      # số chuỗi học song song / sequences in parallel
block_size = 64      # độ dài ngữ cảnh / context length
n_embd     = 96      # số chiều vector / embedding size
n_head     = 4       # số đầu attention / attention heads
n_layer    = 3       # số tầng Transformer / transformer blocks
dropout    = 0.1
lr         = 1e-3
max_iters  = 3000    # tăng để tốt hơn / increase for better output
eval_iters = 20
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
print('device:', device)

## 2. Dữ liệu / Data

Máy chỉ hiểu số → ta **tokenize** char-level: mỗi ký tự = một số nguyên. Cell dưới tự tải tiny-shakespeare (1MB) nếu chưa có.

Computers only understand numbers → char-level **tokenization**: each character = one integer. The cell auto-downloads tiny-shakespeare if missing.

In [ ]:
import os, urllib.request
if not os.path.exists('input.txt'):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, 'input.txt')
    print('downloaded input.txt')

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('length in chars:', len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]        # 'Hi' -> [20, 47]
decode = lambda l: ''.join(itos[i] for i in l) # [20, 47] -> 'Hi'
print('vocab_size:', vocab_size)
print('sample encode:', encode('Hi there'))

In [ ]:
# Tách 90% train / 10% val, rồi tạo hàm lấy lô (batch)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size]     for i in ix])   # đầu vào / input
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])   # đáp án dịch +1 / target shifted by 1
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('input shape :', xb.shape)
print('target shape:', yb.shape)

## 3. Mô hình / Model

GPT = xếp chồng các **Transformer block**. Mỗi block gồm **self-attention** (các token nói chuyện với nhau, chỉ nhìn quá khứ) + **MLP** (mỗi token tự suy nghĩ).

A GPT stacks **Transformer blocks**. Each = **self-attention** (tokens talk, past-only) + **MLP** (per-token thinking).

In [ ]:
class Head(nn.Module):
    """Một đầu self-attention / one self-attention head"""
    def __init__(self, hs):
        super().__init__()
        self.key   = nn.Linear(n_embd, hs, bias=False)
        self.query = nn.Linear(n_embd, hs, bias=False)
        self.value = nn.Linear(n_embd, hs, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5          # điểm tương đồng
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # che tương lai / mask future
        att = self.drop(F.softmax(att, dim=-1))
        return att @ v

In [ ]:
class MultiHead(nn.Module):
    """Nhiều đầu chạy song song / several heads in parallel"""
    def __init__(self):
        super().__init__()
        hs = n_embd // n_head
        self.heads = nn.ModuleList([Head(hs) for _ in range(n_head)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.drop(self.proj(out))

class MLP(nn.Module):
    """Mạng suy nghĩ của từng token / per-token feed-forward"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd), nn.GELU(),
            nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    """1 Transformer block: attention + MLP, có residual (x + ...)"""
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.attn, self.mlp = MultiHead(), MLP()
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # token trao đổi thông tin
        x = x + self.mlp(self.ln2(x))    # token tự xử lý
        return x

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # token -> vector
        self.pos_emb = nn.Embedding(block_size, n_embd)   # vị trí -> vector
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size)      # -> điểm mỗi ký tự
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=device))
        x = self.blocks(tok + pos)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

model = MiniGPT().to(device)
print('So tham so / params: %.2fK' % (sum(p.numel() for p in model.parameters())/1e3))

## 4. Huấn luyện / Train

Vòng lặp: đoán → tính **loss** (sai số) → `backward` tính gradient → `step` chỉnh tham số. Loss giảm = mô hình học tốt lên.

Loop: predict → compute **loss** → `backward` for gradients → `step` to update. Falling loss = learning.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=lr)

@torch.no_grad()
def est_loss():
    model.eval(); out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            _, l = model(*get_batch(split)); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train(); return out

for it in range(max_iters):
    if it % 300 == 0:
        l = est_loss(); print(f"iter {it:4d}: train {l['train']:.3f}, val {l['val']:.3f}")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
print('Xong! / Done.')

## 5. Sinh văn bản / Generate

Cho một ký tự khởi đầu, mô hình đoán ký tự tiếp theo rồi nối vào, lặp lại. / Start token → predict next → append → repeat.

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, 500)[0].tolist()))

## 6. Thử nghiệm / Experiments

- Thay `input.txt` bằng thơ, lời nhạc, hoặc code của bạn → mô hình bắt chước phong cách đó.
- Tăng `max_iters`, `n_layer`, `n_embd` để chất lượng tốt hơn.
- Thêm `temperature` vào `generate` (chia logits) để điều chỉnh độ sáng tạo.

Swap `input.txt` for lyrics/poems/code; raise the size knobs; add `temperature` for creativity control.